# pariwise distance to pairwise corrs. Generate json first

In [ ]:
import json
import numpy as np
import matplotlib.pyplot as plt

# ---- Editable parameters ----
MAX_DIST = 40
BIN_WIDTH = 4.0
MIN_COUNT = 5
# ------------------------------

with open("distance_correlation_data.json") as f:
    data = json.load(f)

fig, ax = plt.subplots(1, 1, figsize=(3, 3))

colors = {
    "baseline": "red",
    "topo":     "black",
}
labels = {
    "baseline": "Baseline\n(Unsorted)",
    "topo":     "Topographic\n(Unsorted)",
}

for key in ["baseline", "topo"]:
    d = np.array(data[key]["distances"])
    c = np.array(data[key]["correlations"])

    mask = d <= MAX_DIST
    d, c = d[mask], c[mask]

    bins = np.arange(0, MAX_DIST + BIN_WIDTH, BIN_WIDTH)
    bin_idx = np.digitize(d, bins)

    bin_centers, means, variances = [], [], []
    for b in range(1, len(bins)):
        m = bin_idx == b
        if m.sum() > MIN_COUNT:
            bin_centers.append((bins[b - 1] + bins[b]) / 2)
            means.append(c[m].mean())
            variances.append(c[m].var())

    bin_centers = np.array(bin_centers)
    means = np.array(means)
    variances = np.array(variances)
    color = colors[key]

    ax.fill_between(
        bin_centers, means - variances, means + variances,
        color=color, alpha=0.2, zorder=1,
    )
    ax.plot(
        bin_centers, means,
        color=color, lw=1.5, zorder=2, label=labels[key],
    )

ax.axhline(0, color='gray', lw=0.5, ls='-', zorder=0)
ax.set_xlim(0, MAX_DIST)
ax.set_xlabel("Pairwise distance\nbetween neurons", fontsize=11, fontweight='bold')
ax.set_ylabel("Pairwise correlation\nof firing patterns", fontsize=11, fontweight='bold')
ax.spines['top'].set_visible(False)
ax.spines['right'].set_visible(False)
ax.spines['left'].set_linewidth(1.2)
ax.spines['bottom'].set_linewidth(1.2)
ax.tick_params(axis='both', labelsize=10, width=1.2)
ax.legend(frameon=False, fontsize=10, loc='upper right')
plt.tight_layout()
plt.savefig("distance_vs_correlation_styled.pdf", dpi=300, bbox_inches='tight')
plt.show()